In [0]:
# Criando o schema da Gold Layer.
# Aqui ficarão os dados prontos para consumo.

spark.sql("""
CREATE SCHEMA IF NOT EXISTS gold
""")

DataFrame[]

In [0]:
from pyspark.sql.functions import sum, avg, max, min, count, round

# 1. Gold - CO2 KPIs por País
df_co2_kpi = (
    spark.table("silver.co2_emissions")
    .groupBy("country")
    .agg(
        round(sum("co2_emissions"), 2).alias("total_emissions"),
        round(avg("co2_emissions"), 2).alias("avg_emissions"),
        round(max("co2_emissions"), 2).alias("max_emissions"),
        round(min("co2_emissions"), 2).alias("min_emissions"),
        count("*").alias("years_count")
    )
    .orderBy("total_emissions", ascending=False)
)

df_co2_kpi.write.format("delta").mode("overwrite").saveAsTable("gold.emissions_by_country")
print(f"✅ Gold - Emissions KPI: {df_co2_kpi.count()} países")

✅ Gold - Emissions KPI: 50 países


In [0]:
# 2. Silver -  Energy Mix
from pyspark.sql.window import Window
from pyspark.sql.functions import col, first, last, round

df_energy_base = (
    spark.table("silver.energy_mix")
    .withColumn(
        "energy_transition_gap",
        col("fossil_pct") - col("renewable_pct")
    )
)

window_country = (
    Window
    .partitionBy("country")
    .orderBy("year")
    .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
)

df_energy_transition = (
    df_energy_base
    .withColumn("first_year", first("year").over(window_country))
    .withColumn("last_year", last("year").over(window_country))
    .withColumn("first_gap", first("energy_transition_gap").over(window_country))
    .withColumn("last_gap", last("energy_transition_gap").over(window_country))
    .withColumn("gap_change", col("last_gap") - col("first_gap"))
    .select(
        "country",
        "first_year",
        "last_year",
        "first_gap",
        "last_gap",
        "gap_change"
    )
    .dropDuplicates()
    .orderBy("gap_change")
)

df_energy_transition.write.format("delta").mode("overwrite").saveAsTable("gold.energy_transition")
print(f"Gold - Energy Transition: {df_energy_transition.count()} países")

Gold - Energy Transition: 50 países


In [0]:
# 3. Gold - Temperature Anomaly por Região
df_temp_kpi = (
    spark.table("silver.temperature_anomaly")
    .groupBy("region")
    .agg(
        round(avg("temp_anomaly"), 2).alias("avg_anomaly"),
        round(max("temp_anomaly"), 2).alias("max_anomaly"),
        round(min("temp_anomaly"), 2).alias("min_anomaly"),
        count("*").alias("months_count")
    )
    .orderBy("avg_anomaly", ascending=False)
)

df_temp_kpi.write.format("delta").mode("overwrite").saveAsTable("gold.temperature_anomaly_by_country")
print(f"✅ Gold - Temperature Anomaly KPI: {df_temp_kpi.count()} regiões")

✅ Gold - Temperature Anomaly KPI: 8 regiões


In [0]:
# 4 .Silver - Climate Events
df_high_severity_events = (
    spark.table("silver.climate_events")
    .withColumn(
        "high_severity_event",
        col("severity_score") >= 7
    )
    .groupBy("year", "region", "event_type")
    .agg(
        count("*").alias("total_events"),
        sum(col("high_severity_event").cast("int")).alias("high_severity_events"),
        round(avg("severity_score"), 2).alias("avg_severity_score")
    )
    .orderBy("year", "region")
)

display(df_high_severity_events)

year,region,event_type,total_events,high_severity_events,avg_severity_score
2003,EUROPE,heatwave,1,1,8.0
2004,ASIA,tsunami,1,1,9.0
2005,USA,hurricane,1,1,9.0
2008,CHINA,earthquake,1,1,8.0
2009,GLOBAL,policy,1,1,8.0
2010,PAKISTAN,flood,1,1,8.0
2010,RUSSIA,heatwave_fire,1,1,8.0
2011,JAPAN,tsunami,1,1,9.0
2012,USA,hurricane,1,1,8.0
2013,PHILIPPINES,typhoon,1,1,8.0


In [0]:
# 5. Gold - Carbon Prices KPIs Temporais
from pyspark.sql.functions import year, col

df_carbon_kpi = (
    spark.table("silver.carbon_prices")
    .groupBy(year(col("date")).alias("year"))
    .agg(
        round(avg("price"), 2).alias("avg_price"),
        round(max("price"), 2).alias("max_price"),
        round(min("price"), 2).alias("min_price"),
        count("*").alias("trading_days")
    )
    .orderBy("year")
)

df_carbon_kpi.write.format("delta").mode("overwrite").saveAsTable("gold.carbon_prices_yearly")
print(f"✅ Gold - Carbon Prices KPI: {df_carbon_kpi.count()} anos")

✅ Gold - Carbon Prices KPI: 22 anos


In [0]:
print("\n" + "="*70)
print("✅ TODAS AS TABELAS GOLD CRIADAS COM SUCESSO!")
print("="*70)


✅ TODAS AS TABELAS GOLD CRIADAS COM SUCESSO!


In [0]:
print("\n" + "="*70)
print("📊 ANÁLISE 1: Top 15 Países Maiores Emissores de CO2")
print("="*70)

display(
    spark.sql("""
    SELECT 
        country,
        ROUND(total_emissions, 0) as total_emissions,
        ROUND(avg_emissions, 0) as avg_emissions,
        years_count
    FROM gold.emissions_by_country
    LIMIT 15
    """)
)


📊 ANÁLISE 1: Top 15 Países Maiores Emissores de CO2


country,total_emissions,avg_emissions,years_count
CHINA,238502.0,8833.0,27
UNITED STATES,142242.0,5268.0,27
INDIA,55023.0,2038.0,27
RUSSIA,45165.0,1673.0,27
JAPAN,31045.0,1150.0,27
GERMANY,19763.0,732.0,27
IRAN,16478.0,610.0,27
SOUTH KOREA,15967.0,591.0,27
CANADA,15374.0,569.0,27
INDONESIA,13619.0,504.0,27


In [0]:
print("⚡ ANÁLISE 2: Países com Maior Dependência de Combustíveis Fósseis")

display(
    spark.sql("""
    SELECT
        country,
        ROUND(AVG(fossil_pct), 2) as avg_fossil_pct,
        ROUND(AVG(renewable_pct), 2) as avg_renewable_pct,
        ROUND(AVG(nuclear_pct), 2) as avg_nuclear_pct,
        COUNT(*) as years_count
    FROM silver.energy_mix
    GROUP BY country
    ORDER BY avg_fossil_pct DESC
    LIMIT 15
    """)
)

⚡ ANÁLISE 2: Países com Maior Dependência de Combustíveis Fósseis


country,avg_fossil_pct,avg_renewable_pct,avg_nuclear_pct,years_count
EGYPT,93.82,5.18,1.0,27
UAE,93.59,5.42,0.99,27
ISRAEL,93.53,5.53,0.95,27
TURKEY,93.29,5.58,1.13,27
IRAQ,93.26,5.67,1.07,27
IRAN,93.24,5.8,0.96,27
KUWAIT,93.17,5.67,1.16,27
QATAR,93.02,5.8,1.17,27
SAUDI ARABIA,92.32,6.48,1.2,27
ALGERIA,92.18,6.48,1.34,27


In [0]:
print("\n" + "="*70)
print("🌡️ ANÁLISE 3: Regiões com Maior Anomalia de Temperatura")
print("="*70)

display(
    spark.sql("""
    SELECT 
        region,
        ROUND(avg_anomaly, 2) as avg_temperature_anomaly,
        ROUND(max_anomaly, 2) as max_temperature_anomaly,
        months_count
    FROM gold.temperature_anomaly_by_country
    ORDER BY avg_anomaly DESC
    LIMIT 15
    """)
)


🌡️ ANÁLISE 3: Regiões com Maior Anomalia de Temperatura


region,avg_temperature_anomaly,max_temperature_anomaly,months_count
ARCTIC,2.73,4.52,316
EUROPE,1.29,2.27,316
NORTH_AMERICA,1.13,2.01,316
NORTHERN_HEMISPHERE,1.02,1.79,316
ASIA,0.97,1.7,316
GLOBAL,0.78,1.46,316
SOUTHERN_HEMISPHERE,0.58,1.13,316
TROPICS,0.51,1.09,316


In [0]:
print("🌍 ANÁLISE 4: Eventos Climáticos por Região e Tipo")

display(
    spark.sql("""
    SELECT
        region,
        event_type,
        COUNT(*) as total_events,
        ROUND(AVG(severity_score), 2) as avg_severity_score,
        MAX(severity_score) as max_severity_score,
        SUM(is_policy) as policy_events,
        SUM(is_extreme_weather) as extreme_weather_events,
        SUM(is_disaster) as disaster_events
    FROM silver.climate_events
    GROUP BY region, event_type
    ORDER BY total_events DESC, avg_severity_score DESC
    LIMIT 20
    """)
)

🌍 ANÁLISE 4: Eventos Climáticos por Região e Tipo


region,event_type,total_events,avg_severity_score,max_severity_score,policy_events,extreme_weather_events,disaster_events
GLOBAL,policy,11,7.91,10,11,0,0
USA,hurricane,4,8.25,9,0,4,0
GLOBAL,temperature,3,8.33,9,0,0,0
PAKISTAN,flood,2,8.5,9,0,2,0
AUSTRALIA,wildfire,2,8.0,8,0,2,0
CARIBBEAN,hurricane,2,8.0,8,0,2,0
USA,wildfire,2,8.0,9,0,2,0
EUROPE,heatwave,2,7.5,8,0,2,0
ASIA,tsunami,1,9.0,9,0,0,1
JAPAN,tsunami,1,9.0,9,0,0,1


In [0]:
print("\n" + "="*70)
print("💰 ANÁLISE 5: Evolução Preço do Carbono por Ano")
print("="*70)

display(
    spark.sql("""
    SELECT 
        year,
        ROUND(avg_price, 2) as avg_price,
        ROUND(max_price, 2) as max_price,
        ROUND(min_price, 2) as min_price,
        trading_days
    FROM gold.carbon_prices_yearly
    ORDER BY year
    """)
)


💰 ANÁLISE 5: Evolução Preço do Carbono por Ano


year,avg_price,max_price,min_price,trading_days
2005,15.18,24.18,7.13,181
2006,24.65,30.84,17.2,260
2007,8.83,17.68,0.05,261
2008,15.99,28.76,0.18,332
2009,10.36,24.1,2.31,522
2010,6.96,13.39,1.79,522
2011,5.85,11.06,1.83,520
2012,5.13,8.52,2.55,522
2013,5.12,10.64,3.44,553
2014,6.95,12.05,4.17,783


In [0]:
print("🔋 ANÁLISE 6: Países com Maior Evolução na Transição Energética")

display(
    spark.sql("""
    SELECT
        country,
        first_year,
        last_year,
        ROUND(first_gap, 2) AS first_gap,
        ROUND(last_gap, 2) AS last_gap,
        ROUND(gap_change, 2) AS gap_change
    FROM gold.energy_transition
    ORDER BY gap_change ASC
    LIMIT 15
    """)
)

🔋 ANÁLISE 6: Países com Maior Evolução na Transição Energética


country,first_year,last_year,first_gap,last_gap,gap_change
UNITED KINGDOM,2000,2026,90.57,40.4,-50.17
GERMANY,2000,2026,82.37,38.21,-44.16
AUSTRALIA,2000,2026,74.97,31.85,-43.12
POLAND,2000,2026,70.1,29.12,-40.98
ITALY,2000,2026,69.01,29.27,-39.74
SPAIN,2000,2026,72.79,36.18,-36.61
BELGIUM,2000,2026,71.86,35.39,-36.47
CZECH REPUBLIC,2000,2026,66.25,30.5,-35.75
DENMARK,2000,2026,70.06,36.46,-33.6
FINLAND,2000,2026,67.04,34.03,-33.01


In [0]:
print("\n" + "="*70)
print("📈 ANÁLISE 7: Emissões Globais por Ano")
print("="*70)

display(
    spark.sql("""
    SELECT 
        year,
        COUNT(DISTINCT country) as countries_count,
        ROUND(SUM(co2_emissions), 0) as total_global_emissions,
        ROUND(AVG(co2_emissions), 0) as avg_emissions_per_country
    FROM silver.co2_emissions
    GROUP BY year
    ORDER BY year DESC
    LIMIT 20
    """)
)


📈 ANÁLISE 7: Emissões Globais por Ano


year,countries_count,total_global_emissions,avg_emissions_per_country
2026,50,35024.0,700.0
2025,50,34853.0,697.0
2024,50,34700.0,694.0
2023,50,34099.0,682.0
2022,50,33376.0,668.0
2021,50,32617.0,652.0
2020,50,31722.0,634.0
2019,50,31941.0,639.0
2018,50,32244.0,645.0
2017,50,32015.0,640.0


In [0]:
print("\n" + "="*70)
print("📊 RESUMO FINAL DO PIPELINE")
print("="*70)

bronze_tables = spark.sql("SHOW TABLES IN bronze").collect()
silver_tables = spark.sql("SHOW TABLES IN silver").collect()
gold_tables = spark.sql("SHOW TABLES IN gold").collect()

print(f"\n🔷 BRONZE LAYER: {len(bronze_tables)} tabelas")
for row in bronze_tables:
    print(f"   - {row.tableName}")

print(f"\n🔸 SILVER LAYER: {len(silver_tables)} tabelas")
for row in silver_tables:
    print(f"   - {row.tableName}")

print(f"\n🟡 GOLD LAYER: {len(gold_tables)} tabelas analíticas")
for row in gold_tables:
    print(f"   - {row.tableName}")

print(f"\n{'='*70}")
print(f"✅ Pipeline Completo Executado com Sucesso!")
print(f"{'='*70}")


📊 RESUMO FINAL DO PIPELINE

🔷 BRONZE LAYER: 6 tabelas
   - carbon_prices_raw
   - climate_events_raw
   - co2_emissions_raw
   - energy_mix_raw
   - sales_raw
   - temperature_anomaly_raw

🔸 SILVER LAYER: 6 tabelas
   - carbon_prices
   - climate_events
   - co2_emissions
   - energy_mix
   - sales_clean
   - temperature_anomaly

🟡 GOLD LAYER: 6 tabelas analíticas
   - carbon_prices_yearly
   - emissions_by_country
   - energy_transition
   - sales_by_category
   - sales_by_region
   - temperature_anomaly_by_country

✅ Pipeline Completo Executado com Sucesso!
